# Are inversions in repeat regions?
Now that we did some repeat modeling and classification, we can see if the breakpoints of any of the simulated inversions fall into these identified repeat regions.

In [2]:
library(dplyr)
library(tidyr)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [3]:
repeats <- read.table("repeat.regions", header = T)
repeats <- repeats[order(repeats$contig), ]
head(repeats)

,contig,position_start,position_end,class
,<chr>,<int>,<int>,<chr>
1,2L,11713984,11714447,Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;Retrotransposon;Long_Terminal_Repeat_Element;Gypsy-ERV;Gypsy
2,2L,11714507,11715213,Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;Retrotransposon;Long_Terminal_Repeat_Element;Gypsy-ERV;Gypsy
3,2L,11715209,11720435,Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;Retrotransposon;Long_Terminal_Repeat_Element;Gypsy-ERV;Gypsy
4,2L,11720465,11720573,Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;Retrotransposon;Long_Terminal_Repeat_Element;Gypsy-ERV;Gypsy
5,2L,1220606,1221069,Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;Retrotransposon;Long_Terminal_Repeat_Element;Gypsy-ERV;Gypsy
6,2L,1221129,1221835,Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;Retrotransposon;Long_Terminal_Repeat_Element;Gypsy-ERV;Gypsy


We'll to read in the inventory of all inversion positions and pull out the unique inversions. I really should have committed to either comma or tab separated tables.

In [4]:
inversions <- read.csv("../assess_called_sv/linkedread.sv.assessment", header = T)
inversions <- unique(inversions[, c(1,6,10,11)])
rownames(inversions) <- NULL
inversions$is_repeat <- as.character(NA)
head(inversions)
nrow(inversions)

,contig,size,position_start,position_end,is_repeat
,<chr>,<chr>,<int>,<int>,<chr>
1,2L,small,3193196,3214221,NA
2,2L,small,3940211,3944863,NA
3,2L,small,13024404,13026577,NA
4,2L,small,14845709,14861200,NA
5,2L,small,20107901,20113695,NA
6,2R,small,4328632,4333509,NA


[1] 48

To do the matchy-matchy, we can borrow the fuzzy matching function we used to assess inversion calling. It will need some modifications to accommodate the slightly different use case.

In [5]:
for(i in 1:nrow(inversions)){
    .row <- inversions[i,]
    query <- which(
        repeats$contig == .row$contig &
        (repeats$position_start - 150 <= .row$position_start & repeats$position_end + 150 >= .row$position_start) |
        (repeats$position_start - 150 <= .row$position_end) & (repeats$position_end + 150 >= .row$position_end)
    )
    if(length(query) > 0) {
        inversions$is_repeat[i] <- do.call("paste", as.list(repeats$class[query]))
    }
}

In [6]:
are_repeats <- inversions[!is.na(inversions$is_repeat),]
are_repeats

,contig,size,position_start,position_end,is_repeat
,<chr>,<chr>,<int>,<int>,<chr>
1,2L,small,3193196,3214221,Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;Retrotransposon;Long_Terminal_Repeat_Element;Gypsy-ERV;Gypsy Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;LINE;Group-II;Group-2;R1-like;CR1-group;CR1
2,2L,small,3940211,3944863,Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;Retrotransposon;Long_Terminal_Repeat_Element;Gypsy-ERV;Gypsy
5,2L,small,20107901,20113695,Interspersed_Repeat;Transposable_Element;Class_II_DNA_Transposition;Transposase;CACTA;Transib Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;Retrotransposon;Long_Terminal_Repeat_Element;Ty1-Copia
6,2R,small,4328632,4333509,Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;LINE;Group-II;Group-2;R1-like;R1-group;I-group;Jockey Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;Retrotransposon;Long_Terminal_Repeat_Element;Gypsy-ERV;Gypsy
16,3R,small,20006864,20010483,Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;Retrotransposon;Long_Terminal_Repeat_Element;Gypsy-ERV;Gypsy
21,2L,medium,1591146,1776073,Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;Retrotransposon;Long_Terminal_Repeat_Element;Gypsy-ERV;Gypsy Interspersed_Repeat;Transposable_Element;Class_II_DNA_Transposition;Transposase;CACTA;Transib Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;Retrotransposon;Long_Terminal_Repeat_Element;Gypsy-ERV;Gypsy
25,2R,medium,9640874,9978281,Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;Retrotransposon;Long_Terminal_Repeat_Element;Gypsy-ERV;Gypsy
33,3R,medium,16760480,16905871,Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;Retrotransposon;Long_Terminal_Repeat_Element;Bel-Pao
34,3R,medium,22722816,22831351,Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;Retrotransposon;Long_Terminal_Repeat_Element;Gypsy-ERV;Gypsy


Let's find which inversions had the most false negatives.

In [7]:
called_sv <- read.csv("../assess_called_sv/linkedread.sv.assessment", header = T)
false_neg <- filter(called_sv, assessment == "false negative") %>%
    group_by(contig, position_start, size) %>%
    summarize(n = length(depth)) %>%
    arrange(desc(n))
false_neg$is_repeat <- FALSE
head(false_neg, 20)

`summarise()` has grouped output by 'contig', 'position_start'. You can
override using the `.groups` argument.


contig,position_start,size,n,is_repeat
<chr>,<int>,<chr>,<int>,<lgl>
2L,1591146,medium,55,FALSE
2L,6436329,medium,55,FALSE
2R,4328632,small,55,FALSE
3R,16760480,medium,40,FALSE
3R,26216293,medium,30,FALSE
2R,9640874,medium,26,FALSE
2R,17910285,medium,26,FALSE
2L,961650,xl,25,FALSE
2R,15862083,medium,25,FALSE


We can now combine the inversions-in-repeats with the inversions-not-detected dataframe to see if there's a correlation of hard-to-call inversions being associated with repeat regions.

In [28]:
for(i in 1:nrow(false_neg)){
    .row <- false_neg[i,]
    query <- which(
        are_repeats$contig == .row$contig &
        are_repeats$position_start == .row$position_start
    )
    if(length(query) > 0) {
        false_neg$is_repeat[i] <- TRUE
    }
}
false_neg$platform <- "linkedread"
head(arrange(false_neg, desc(is_repeat)))

contig,position_start,size,n,is_repeat,platform
<chr>,<int>,<chr>,<int>,<lgl>,<chr>
2L,1591146,medium,55,TRUE,linkedread
2R,4328632,small,55,TRUE,linkedread
3R,16760480,medium,40,TRUE,linkedread
3R,26216293,medium,30,TRUE,linkedread
2R,9640874,medium,26,TRUE,linkedread
2L,20107901,small,20,TRUE,linkedread


For comparison sake, we need to do this with the long-read inversion detections as well. We'll read it in and extract only the relevant columns and only rows where `assessment` is `false negative`.

In [22]:
long_inv <- read.csv("../longread_workflow/longread.sv.assessment", header = T)
false_neg_long <- filter(long_inv[,c(2:5, 8,10)], assessment == "false negative") %>%
    group_by(contig, position_start, size, platform) %>%
    summarize(n = length(assessment)) %>%
    arrange(desc(n))
false_neg_long$is_repeat <- FALSE
head(false_neg_long, 20)

`summarise()` has grouped output by 'contig', 'position_start', 'size'. You can
override using the `.groups` argument.


contig,position_start,size,platform,n,is_repeat
<chr>,<int>,<chr>,<chr>,<int>,<lgl>
2R,4328632,small,ontshort,39,FALSE
2R,4328632,small,pacbio,24,FALSE
3R,16760480,medium,ontshort,24,FALSE
2R,4328632,small,ontlong,22,FALSE
2R,18236490,medium,ontshort,22,FALSE
2L,961650,xl,ontshort,21,FALSE
2R,10569437,small,ontshort,21,FALSE
2R,9640874,medium,ontshort,20,FALSE
2R,9952840,small,ontlong,20,FALSE


Now do the same matching between inversions and repeat regions

In [25]:
for(i in 1:nrow(false_neg_long)){
    .row <- false_neg_long[i,]
    query <- which(
        are_repeats$contig == .row$contig &
        are_repeats$position_start == .row$position_start
    )
    if(length(query) > 0) {
        false_neg_long$is_repeat[i] <- TRUE
    }
}
head(arrange(false_neg_long, desc(is_repeat)))

contig,position_start,size,platform,n,is_repeat
<chr>,<int>,<chr>,<chr>,<int>,<lgl>
2R,4328632,small,ontshort,39,TRUE
2R,4328632,small,pacbio,24,TRUE
3R,16760480,medium,ontshort,24,TRUE
2R,4328632,small,ontlong,22,TRUE
2R,9640874,medium,ontshort,20,TRUE
2R,9640874,medium,ontlong,18,TRUE


Let's combine the two dataframes so we can start plotting them to meaningfully represent these data. This will also include making `platform`, `size`, and `contig` ordered factors.

In [32]:
false_neg_all <- rbind(false_neg, false_neg_long) %>% arrange(contig, position_start, size, platform, n)
false_neg_all$contig <- factor(false_neg_all$contig, levels = c("2L", "2R","3L","3R"), ordered = T)
false_neg_all$size <- factor(false_neg_all$size, levels = c("small", "medium", "large", "xl"), ordered = T)
false_neg_all$platform <- factor(false_neg_all$platform, levels = c("linkedread","ontshort","ontlong","pacbio"), ordered = T)

head(false_neg_all)

contig,position_start,size,n,is_repeat,platform
<ord>,<int>,<ord>,<int>,<lgl>,<ord>
2L,961650,xl,25,FALSE,linkedread
2L,961650,xl,11,FALSE,ontlong
2L,961650,xl,21,FALSE,ontshort
2L,961650,xl,11,FALSE,pacbio
2L,1591146,medium,55,TRUE,linkedread
2L,1591146,medium,11,TRUE,ontlong


It may be worth adding a column of unique identifiers for the inversions for plotting purposes.

In [39]:
false_neg_all <- group_by(false_neg_all, contig, position_start) %>%
    mutate(id = cur_group_id())
head(false_neg_all, 15)

contig,position_start,size,n,is_repeat,platform,id
<ord>,<int>,<ord>,<int>,<lgl>,<ord>,<int>
2L,961650,xl,25,FALSE,linkedread,1
2L,961650,xl,11,FALSE,ontlong,1
2L,961650,xl,21,FALSE,ontshort,1
2L,961650,xl,11,FALSE,pacbio,1
2L,1591146,medium,55,TRUE,linkedread,2
2L,1591146,medium,11,TRUE,ontlong,2
2L,1591146,medium,14,TRUE,ontshort,2
2L,1591146,medium,14,TRUE,pacbio,2
2L,3193196,small,15,TRUE,linkedread,3
